# Online Retail – Data Cleaning Pipeline

**Goal:** Prepare 541,909 rows for EDA.  
**Key choices:** kept returns (negative qty) and free items (£0).

# 1. Import Libraries

In [322]:
import pandas as pd

# 2. Load Dataset

In [323]:
df = pd.read_csv("../data/raw/online_retail_data.csv")

# 3. Dataset Overview

### 3.1 Dataset Shape

In [324]:
df.shape

(541909, 8)

### 3.2 Dataset Information

In [325]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


### 3.3 Statistical Summary

In [326]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


### 3.4 Column Names

In [327]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

### 3.5 First Five Records

In [328]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# 4. Data Quality Assessment

### 4.1 Missing Values

### *Missing Value Count*

In [329]:
df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

### *Missing Value Percentage*

In [330]:
perc_miss = df.isna().sum() / len(df)
perc_miss * 100

InvoiceNo       0.000000
StockCode       0.000000
Description     0.268311
Quantity        0.000000
InvoiceDate     0.000000
UnitPrice       0.000000
CustomerID     24.926694
Country         0.000000
dtype: float64

### 4.2 Duplicate Rows

In [331]:
df.duplicated().sum()

np.int64(5268)

### 4.3 Data Types

In [332]:
df.dtypes

InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

### 4.4 Unique Values

In [333]:
df.nunique()

InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
dtype: int64

### 4.5 Initial Anomaly Detection

### *Negative & Zero Quantity*

In [334]:
# Negative quantities represent returns – retained for return rate analysis.
df[df["Quantity"] < 0].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [335]:
len(df[df["Quantity"] < 0])

10624

In [336]:
# No rows with Quantity = 0 found.
df[df["Quantity"] == 0].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


### *Negative & Zero Unit Price*

In [337]:
# Only 2 negative unit price rows – "adjust bad debt" entries – will be removed.
df[df["UnitPrice"] < 0].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [338]:
# Zero-priced items: samples, promos, damaged replacements, adjustments.
# Retained, but excluded from revenue calculations later.
df[df["UnitPrice"] == 0].sample(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
446552,574906,85034A,damaged,-81,2011-11-07 15:30:00,0.0,NaN,United Kingdom
274544,560923,22777,NaN,-14,2011-07-22 08:54:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
535321,581198,22025,check,-30,2011-12-07 18:26:00,0.0,NaN,United Kingdom
140282,548395,22194,NaN,-9,2011-03-30 17:20:00,0.0,NaN,United Kingdom
87404,543654,22525,NaN,-57,2011-02-11 10:29:00,0.0,NaN,United Kingdom
519355,580145,22925,AMAZON,1,2011-12-02 10:03:00,0.0,NaN,United Kingdom
422750,573114,20713,wrongly coded 23343,1000,2011-10-27 15:36:00,0.0,NaN,United Kingdom
355335,567921,21089,damaged,-7,2011-09-22 17:24:00,0.0,NaN,United Kingdom
369983,569121,21244,NaN,-38,2011-09-30 13:03:00,0.0,NaN,United Kingdom


In [339]:
df[df["UnitPrice"] == 0].sample(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
108687,545565,47563B,NaN,-10,2011-03-03 16:14:00,0.0,NaN,United Kingdom
173265,551666,84750A,NaN,-10,2011-05-03 12:35:00,0.0,NaN,United Kingdom
78425,542880,22242,NaN,10,2011-02-01 12:53:00,0.0,NaN,United Kingdom
150158,549348,84251B,NaN,-60,2011-04-08 11:13:00,0.0,NaN,United Kingdom
192289,553394,15058A,wet/rusty,-30,2011-05-16 16:47:00,0.0,NaN,United Kingdom
146597,548997,84507B,NaN,20,2011-04-05 14:33:00,0.0,NaN,United Kingdom
514181,579681,21615,check,-26,2011-11-30 13:33:00,0.0,NaN,United Kingdom
150623,549497,21527,NaN,3,2011-04-08 15:06:00,0.0,NaN,United Kingdom
468299,576414,17012F,check,14,2011-11-15 11:21:00,0.0,NaN,United Kingdom
499156,578628,23541,NaN,20,2011-11-24 15:55:00,0.0,NaN,United Kingdom


In [340]:
len(df[df["UnitPrice"] < 0])

2

# 5. Data Cleaning

### 5.1 Column Names & Duplicate Rows

In [341]:
# Standardise all column names to lowercase
df.columns = df.columns.str.lower()

In [342]:
df.columns

Index(['invoiceno', 'stockcode', 'description', 'quantity', 'invoicedate',
       'unitprice', 'customerid', 'country'],
      dtype='object')

In [343]:
# Remove duplicate rows
df = df.drop_duplicates()

### 5.2 Invoice Number

In [344]:
# No missing values detected.

### 5.2.1 Cancelled Invoices

Invoices starting with "C" are cancellations – kept to track return history.

In [345]:
cancelled = df[df['invoiceno'].astype(str).str.startswith('C')]
print(f"Cancelled invoice rows: {len(cancelled)}")
print(f"Unique cancelled invoices: {cancelled['invoiceno'].nunique()}")

Cancelled invoice rows: 9251
Unique cancelled invoices: 3836


### 5.3 Stock Code

In [346]:
# No missing values detected.

### 5.4 Description

In [347]:
# Clean: lowercase, spaces → underscores, strip special chars
df['description'] = (
    df['description']
    .str.lower()
    .str.replace(" ", "_")
    .str.strip("_. ")
)

In [348]:
# Drop missing descriptions
df = df[df['description'].notna()]

In [349]:
df['description'].sample(20)

298590               jumbo_bag_vintage_leaf
224104               6_ribbons_rustic_charm
443268     set_of_3_cake_tins_pantry_design
531485      set_of_2_ceramic_painted_hearts
173340                 wrap_alphabet_design
55772             triangular_pouffe_vintage
133612    blue_savannah_picnic_hamper_for_2
273891       assorted_tutti_frutti_bracelet
120452         heart_decoration_with_pearls
52486       easter_decoration_hanging_bunny
384749        traditional_christmas_ribbons
211428            area_patrolled_metal_sign
294856     vintage_cream_cat_food_container
247545               strawberry_shopper_bag
393837         full_english_breakfast_plate
294684              egg_cup_natural_chicken
338787            pin_cushion_babushka_pink
514315    vintage_cream_3_basket_cake_stand
532647                   doormat_union_flag
515021       buffalo_bill_treasure_book_box
Name: description, dtype: object

### 5.5 Quantity
> Kept negative values – they represent returns (analysed later via `is_return` flag).

In [350]:
# Retained negative quantities (returns).

### 5.6 Invoice Date

In [351]:
df['invoicedate'] = pd.to_datetime(df['invoicedate'])

### 5.7 Unit Price
> Kept zero prices (samples/promos). Removed only negative prices (2 rows of "adjust bad debt").

In [352]:
df = df[df["unitprice"] >= 0]

### 5.8 Customer ID

In [353]:
# Convert to nullable Int64 to preserve missing values as <NA>
df['customerid'] = df['customerid'].astype('Int64')

### 5.9 Country

In [354]:
# Standardise: lowercase + underscores
df['country'] = (
    df['country']
    .str.lower()
    .str.replace(" ", "_")
)

# 6. Final Data Validation

### 6.1 Dataset Shape

In [355]:
df.shape

(535185, 8)

### 6.2 Dataset Information

In [356]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 535185 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   invoiceno    535185 non-null  object        
 1   stockcode    535185 non-null  object        
 2   description  535185 non-null  object        
 3   quantity     535185 non-null  int64         
 4   invoicedate  535185 non-null  datetime64[ns]
 5   unitprice    535185 non-null  float64       
 6   customerid   401604 non-null  Int64         
 7   country      535185 non-null  object        
dtypes: Int64(1), datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 37.3+ MB


### 6.3 Missing Values

In [357]:
df.isna().sum()

invoiceno           0
stockcode           0
description         0
quantity            0
invoicedate         0
unitprice           0
customerid     133581
country             0
dtype: int64

### 6.4 Duplicate Rows

In [358]:
df.duplicated().sum()

np.int64(0)

## Cleaning Summary

| Step | Action | Rows Remaining |
| :--- | :--- | ---: |
| Start | Raw data | 541,909 |
| 1 | Removed duplicates | 536,641 |
| 2 | Dropped missing descriptions | 535,187 |
| 3 | Removed negative unit prices | 535,185 |
| **Final** | **Clean dataset** | **535,185** |

### 6.5 Sample Records

In [359]:
df.sample(10)

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
127933,547248,21012,antique_all_glass_candlestick,10,2011-03-22 09:23:00,2.46,<NA>,united_kingdom
274519,560921,22440,balloon_water_bomb_pack_of_35,20,2011-07-21 19:17:00,0.42,16224,united_kingdom
214059,555564,82494L,wooden_frame_antique_white,4,2011-06-05 15:01:00,2.95,15005,united_kingdom
138502,548198,22994,travel_card_wallet_retrospot,1,2011-03-29 16:05:00,0.42,16712,united_kingdom
516235,579885,22260,felt_egg_cosy_blue_rabbit,1,2011-11-30 17:37:00,0.85,15444,united_kingdom
284931,561893,21114,lavender_scented_fabric_heart,15,2011-07-31 14:39:00,1.25,12942,united_kingdom
61174,541423,21946,party_time_design_flannel,1,2011-01-17 17:54:00,1.63,<NA>,united_kingdom
274590,560926,21430,set/3_red_gingham_rose_storage_box,2,2011-07-22 09:20:00,7.46,<NA>,united_kingdom
480491,577303,85099B,jumbo_bag_red_retrospot,10,2011-11-18 13:02:00,2.08,14390,united_kingdom
165922,550835,22804,candleholder_pink_hanging_heart,1,2011-04-21 10:52:00,2.95,15034,united_kingdom


In [360]:
df.isna().sum()

invoiceno           0
stockcode           0
description         0
quantity            0
invoicedate         0
unitprice           0
customerid     133581
country             0
dtype: int64

In [361]:
df.duplicated().sum()

np.int64(0)

In [362]:
df.to_csv("../data/processed/online_retail_data_clean.csv", index=False)

## Next Steps

Saved to `online_retail_data_clean.csv`.  
Proceed to `02_exploratory_data_analysis.ipynb` for visualisation and recommendations.